# 🤖 Modelado Supervisado — FIDE Chess Dataset

**Evaluación 2 — Implementación Supervisada (20%)**

Este notebook cubre:
1. Carga del dataset preprocesado
2. Preparación de features y split train/test
3. Entrenamiento de múltiples modelos de clasificación
4. Comparación visual de rendimiento
5. Justificación de decisiones técnicas

In [ ]:
%load_ext kedro.ipython

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)
RANDOM_STATE = 42

## 1. Carga del Dataset Preprocesado

Usamos el dataset que ya pasó por los pipelines de ingesta, limpieza y transformación.

In [ ]:
# Cargar el dataset transformado desde el catálogo Kedro
df = catalog.load('fide_preprocessed_data')
print(f'Dataset preprocesado: {df.shape}')
display(df.head())

## 2. Preparación de Features

**Variable objetivo:** `is_expert` (1 si ELO promedio > 2000, 0 en caso contrario)

**Justificación:** Elegimos un problema de clasificación binaria porque permite evaluar si un jugador alcanza el nivel de experto, lo cual es un umbral reconocido por la FIDE y tiene significado práctico en el dominio.

In [ ]:
# Seleccionar features
FEATURE_COLS = ['rating_std_avg', 'rating_change', 'total_months_active',
                'age_approx', 'gender_encoded', 'title_encoded']
TARGET = 'is_expert'

available = [c for c in FEATURE_COLS if c in df.columns]
ml_data = df[available + [TARGET]].dropna()

print(f'Features disponibles: {available}')
print(f'Dataset para ML: {ml_data.shape}')
print(f'\nDistribución del target:')
print(ml_data[TARGET].value_counts())
print(f'\nBalance: {ml_data[TARGET].mean():.2%} expertos')

In [ ]:
# Split train/test estratificado
X = ml_data[available]
y = ml_data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train target balance: {y_train.mean():.2%}')
print(f'Test target balance:  {y_test.mean():.2%}')

## 3. Entrenamiento de Múltiples Modelos

Entrenamos 5 modelos de clasificación diferentes para comparar su rendimiento:

| Modelo | Justificación |
|--------|---------------|
| Logistic Regression | Baseline lineal, interpretable |
| Random Forest | Ensemble robusto, maneja no-linealidad |
| KNN | No paramétrico, útil para comparar |
| SVM | Buen rendimiento en espacios de alta dimensión |
| Gradient Boosting | Estado del arte en datos tabulares |

In [ ]:
# Definir modelos
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'SVM (RBF)': SVC(kernel='rbf', random_state=RANDOM_STATE, probability=True),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

# Entrenar y evaluar
results = {}
for name, model in models.items():
    print(f'Entrenando {name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    results[name] = {'accuracy': acc, 'f1_score': f1, 'model': model}
    
    print(f'  Accuracy: {acc:.4f}, F1: {f1:.4f}')

print('\nEntrenamiento completado.')

## 4. Comparación Visual de Modelos

In [ ]:
# Tabla comparativa
comparison_df = pd.DataFrame({
    name: {'Accuracy': r['accuracy'], 'F1-Score': r['f1_score']}
    for name, r in results.items()
}).T.sort_values('F1-Score', ascending=False)

display(comparison_df.style.format('{:.4f}').background_gradient(cmap='YlGn'))

# Gráfico de barras
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

comparison_df['Accuracy'].plot.barh(ax=axes[0], color=sns.color_palette('viridis', len(comparison_df)))
axes[0].set_title('Accuracy por Modelo')
axes[0].set_xlim(0, 1)

comparison_df['F1-Score'].plot.barh(ax=axes[1], color=sns.color_palette('magma', len(comparison_df)))
axes[1].set_title('F1-Score por Modelo')
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Reporte detallado del mejor modelo
best_name = comparison_df.index[0]
best_model = results[best_name]['model']
y_pred_best = best_model.predict(X_test)

print(f'\n=== Reporte del Mejor Modelo: {best_name} ===')
print(classification_report(y_test, y_pred_best, zero_division=0))

## 5. Conclusiones del Modelado Supervisado

- Se entrenaron 5 modelos de clasificación diferentes sobre el dataset FIDE preprocesado.
- La variable objetivo `is_expert` permite clasificar jugadores según su nivel de rating.
- Los modelos de ensemble (Random Forest, Gradient Boosting) tienden a obtener mejores resultados.
- En el siguiente notebook se profundizará en la evaluación con métricas avanzadas.